# 🏆 Churn Prediction — Part 1: Baseline Models
## Kaggle Playground S6E3

**Goal:** Establish baseline scores that all future models must beat.

| Model | Type | Expected AUC | Why |
|-------|------|-------------|-----|
| Logistic Regression | Linear | ~0.910 | Simplest possible model |
| Random Forest | Bagging | ~0.910 | Non-linear but no boosting |

**Input:** `X_train.csv`, `X_test.csv`, `y_train.csv`, `test_ids.csv` from Phase 5  
**Output:** `part1_oof_lr.npy`, `part1_oof_rf.npy`, `part1_test_lr.npy`, `part1_test_rf.npy`


In [2]:
import pandas as pd, numpy as np, time, warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

# Load data
X_train = pd.read_csv('/Users/parveenkumarsharma/Downloads/playground-series-s6e3/X_train.csv.csv')
X_test = pd.read_csv('/Users/parveenkumarsharma/Downloads/playground-series-s6e3/X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
test_ids = pd.read_csv('test_ids.csv').squeeze()

N_FOLDS = 5
SEED = 42
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print(f"Data: {X_train.shape[0]:,} train × {X_train.shape[1]} features | {X_test.shape[0]:,} test")
print(f"Target: {y_train.mean()*100:.1f}% churn | CV: Stratified {N_FOLDS}-Fold")


Data: 594,194 train × 71 features | 254,655 test
Target: 22.5% churn | CV: Stratified 5-Fold


In [4]:
# ============================================================
# UNIVERSAL CV FUNCTION (reused in ALL notebooks)
# ============================================================
def run_cv(name, train_fn, X, y, X_test, skf):
    oof = np.zeros(len(y))
    test_preds = np.zeros(len(X_test))
    scores = []
    t0 = time.time()
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        
        model = train_fn(X_tr, y_tr, X_va, y_va)
        
        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        test_preds += model.predict_proba(X_test)[:, 1] / N_FOLDS
        
        fold_auc = roc_auc_score(y_va, oof[va_idx])
        scores.append(fold_auc)
        print(f"  Fold {fold+1}: AUC = {fold_auc:.6f}")
    
    mean_auc = np.mean(scores)
    std_auc = np.std(scores)
    print(f"  ─────────────────────────")
    print(f"  ✅ {name}: {mean_auc:.6f} ± {std_auc:.6f} ({time.time()-t0:.1f}s)")
    return oof, test_preds, mean_auc


## Model 1: Logistic Regression (Baseline)
- Linear model with sigmoid activation
- **Needs StandardScaler** (sensitive to feature magnitudes)
- `class_weight='balanced'` handles imbalance
- `C=0.1` → moderate regularization


In [5]:
# ============================================================
# MODEL 1: LOGISTIC REGRESSION
# ============================================================
scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_sc = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

def train_lr(X_tr, y_tr, X_va, y_va):
    m = LogisticRegression(C=0.1, class_weight='balanced', max_iter=1000, random_state=SEED)
    m.fit(X_tr, y_tr)
    return m

print("▶ LOGISTIC REGRESSION")
oof_lr, test_lr, score_lr = run_cv("LR", train_lr, X_train_sc, y_train, X_test_sc, skf)


▶ LOGISTIC REGRESSION
  Fold 1: AUC = 0.911293
  Fold 2: AUC = 0.912688
  Fold 3: AUC = 0.911759
  Fold 4: AUC = 0.912786
  Fold 5: AUC = 0.909903
  ─────────────────────────
  ✅ LR: 0.911686 ± 0.001054 (20.1s)


## Model 2: Random Forest (Bagging)
- 500 decision trees, each on random data subset
- `max_depth=15` prevents overfitting
- `min_samples_leaf=30` keeps leaves meaningful
- **No scaling needed** (tree-based)


In [6]:
# ============================================================
# MODEL 2: RANDOM FOREST
# ============================================================
def train_rf(X_tr, y_tr, X_va, y_va):
    m = RandomForestClassifier(
        n_estimators=500, max_depth=15, min_samples_leaf=30,
        max_features='sqrt', class_weight='balanced',
        random_state=SEED, n_jobs=-1
    )
    m.fit(X_tr, y_tr)
    return m

print("\n▶ RANDOM FOREST")
oof_rf, test_rf, score_rf = run_cv("RF", train_rf, X_train, y_train, X_test, skf)



▶ RANDOM FOREST
  Fold 1: AUC = 0.913907
  Fold 2: AUC = 0.915171
  Fold 3: AUC = 0.914543
  Fold 4: AUC = 0.915667
  Fold 5: AUC = 0.913036
  ─────────────────────────
  ✅ RF: 0.914465 ± 0.000927 (911.5s)


In [8]:
# ============================================================
# SAVE OOF + TEST PREDICTIONS (for Part 5 ensemble)
# ============================================================
np.save('part1_oof_lr.npy', oof_lr)
np.save('part1_oof_rf.npy', oof_rf)
np.save('part1_test_lr.npy', test_lr)
np.save('part1_test_rf.npy', test_rf)

print("\n" + "=" * 50)
print("  PART 1 RESULTS")
print("=" * 50)
print(f"  Logistic Regression: {score_lr:.6f}")
print(f"  Random Forest:       {score_rf:.6f}")
print(f"\n  ✅ Saved: part1_oof_*.npy, part1_test_*.npy")
print(f"  → These will be used in Part 5 for final ensemble")



  PART 1 RESULTS
  Logistic Regression: 0.911686
  Random Forest:       0.914465

  ✅ Saved: part1_oof_*.npy, part1_test_*.npy
  → These will be used in Part 5 for final ensemble
